In [ ]:
import pandas as pd
import dask.dataframe as dd
import glob
import yaml
from tqdm import tqdm
import os

In [ ]:
import os
os.chdir('../../../../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

In [ ]:
fp = os.path.join(dataset_config['path_cnpat'], 'CNpatents_universe_1985_2025.csv')
cols_all = pd.read_csv(fp, nrows=0, encoding='utf-8-sig')
list(cols_all.columns)

In [ ]:
fp = os.path.join(dataset_config['path_cnpat'], 'CNpatents_universe_1985_2025.csv')

cols = ['专利类型', '申请人', '申请日', '申请号', '申请人类型', '授权公告年份', '申请人地区', '统一社会信用代码']

cnpat = pd.read_csv(
    fp,
    usecols=cols
)

cnpat = cnpat[cnpat['专利类型'] == '实用新型']
cnpat = cnpat[cnpat['申请人类型'] == '企业']
cnpat = cnpat[cnpat['申请人地区'] == '中国']

cnpat

In [ ]:
cnpat['申请日'] = pd.to_datetime(cnpat['申请日'], errors='coerce')
cnpat['申请年'] = cnpat['申请日'].dt.year
cnpat_2010 = cnpat[cnpat['申请日'].dt.year >= 2010]
cnpat_2010 = cnpat_2010.rename(columns={'授权公告年份': '授权年'})
cnpat_2010['grant'] = cnpat_2010['授权年'].notna().astype(int)
cnpat_2010

In [ ]:
cnpat_2010 = cnpat_2010.drop(columns=['专利类型', '申请人类型', '申请日'])
cnpat_2010

In [ ]:
cnpat_2010['申请人'] = cnpat_2010['申请人'].astype(str).str.replace('；', ';', regex=False)
cnpat_2010['统一社会信用代码'] = cnpat_2010['统一社会信用代码'].astype(str).str.replace('；', ';', regex=False)

cnpat_2010['申请人'] = cnpat_2010['申请人'].str.split(';').str[0].str.strip()
cnpat_2010['统一社会信用代码'] = cnpat_2010['统一社会信用代码'].str.split(';').str[0].str.strip()

cnpat_2010


In [ ]:
cnpat_2010.to_csv(dataset_config['path_processed'] + 'CN_CN/CNfirm_utilitymodel_2010_2025.csv', index=False)